# Real RGB-D Semantic Gaussian Budget Experiment

Dataset: **TUM RGB-D freiburg1_desk**, real office RGB-D. Input groundtruth camera poses; panoptic predictions from Mask2Former. 120 associated frames (every fifth source frame), one in six held out from fusion and optimization. Compare CPU fusion, CUDA refinement with 6,000/24,000 splat caps, and freezing ablation. A single exploratory run, not a complete SLAM or semantic accuracy benchmark.

The project source is embedded below. This notebook needs no second upload. Run sequentially on GPU. No Drive mount or public sharing is used. CUDA smoke tests must pass before real experiments.

In [ ]:
#@title Unpack experiment source and inspect GPU
import base64,io,zipfile,os,sys,subprocess,json,time
from pathlib import Path
import torch
print('GPU:',torch.cuda.get_device_name() if torch.cuda.is_available() else 'unavailable')
assert torch.cuda.is_available(), 'Select a GPU runtime'
project=Path('/content/semantic-gaussian-real'); project.mkdir(exist_ok=True)
payload='...'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(payload))) as z:
    for item in z.infolist(): assert (project/item.filename).resolve().is_relative_to(project.resolve())
    z.extractall(project)
os.chdir(project)
os.environ['MAX_JOBS']='2'
print('Project:',project)


In [ ]:
#@title Install dependencies and save the exact environment
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.[segmentation]','gsplat>=1.5,<2','ninja','scikit-image','matplotlib'],check=True)
import platform,shutil
Path('results').mkdir(exist_ok=True)
environment={'python':platform.python_version(),'torch':torch.__version__,'cuda':torch.version.cuda,
             'gpu':torch.cuda.get_device_name(),'nvcc':shutil.which('nvcc')}
Path('results/environment.json').write_text(json.dumps(environment,indent=2))
Path('results/pip-freeze.txt').write_bytes(subprocess.check_output([sys.executable,'-m','pip','freeze']))
print(environment)
def run(*args):
    # Stream child stdout/stderr through the notebook so failures are diagnosable.
    command=[sys.executable,*args]
    with subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as child:
        for line in child.stdout: print(line,end='',flush=True)
        code=child.wait()
        if code: raise subprocess.CalledProcessError(code,command)


In [ ]:
#@title Verify CPU invariants and CUDA forward/backward
run('-m','unittest','discover','-s','tests','-v')
run('-m','sglab.gpu_smoke')
run('-m','sglab','run','--frames','12','--refine','--max-gaussians','8000','--output','results/gpu-smoke')


In [ ]:
#@title Download real TUM desk RGB-D and make the generic dataset
import urllib.request,tarfile
raw=Path('/content/tum-desk'); raw.mkdir(exist_ok=True)
archive=raw/'desk.tgz'
url='https://cvg.cit.tum.de/rgbd/dataset/freiburg1/rgbd_dataset_freiburg1_desk.tgz'
if not archive.exists():
    print('Downloading real TUM desk RGB-D...',flush=True)
    urllib.request.urlretrieve(url,archive)
with tarfile.open(archive) as t: t.extractall(raw,filter='data')
run('-m','sglab','import-tum','--dataset',str(raw/'rgbd_dataset_freiburg1_desk'),
    '--output','data/tum-desk','--frames','120','--every','5')
manifest='data/tum-desk/manifest.json'
print('Associated input frames:',len(json.loads(Path(manifest).read_text())['frames']))


In [ ]:
#@title Predict real semantic and object-instance masks on the GPU
panoptic='data/tum-desk/panoptic.json'
run('-m','sglab','segment','--manifest',manifest,'--output',panoptic,'--device','cuda')
m=json.loads(Path(panoptic).read_text())
import numpy as np
seg_times=[f['segmentation_ms'] for f in m['frames']]
print('Separately cached panoptic inference p50 / p95 ms:',np.percentile(seg_times,[50,95]))


In [ ]:
#@title Run four real-data variants in independent processes
variants=[('cpu-fusion-24000',24000,['--no-refine']),
          ('cuda-cap-6000',6000,[]),
          ('cuda-cap-24000',24000,[]),
          ('cuda-no-freeze-24000',24000,['--no-freeze'])]
for name,cap,flags in variants:
    print('VARIANT:',name,flush=True)
    run('-m','sglab.real_experiment','--manifest',panoptic,'--output',f'results/{name}','--cap',str(cap),*flags)
print('ALL_REAL_VARIANTS_COMPLETE')
print('Recording replay separately; excluded from latency comparisons')
run('-m','sglab.real_experiment','--manifest',panoptic,'--output','results/cuda-replay-24000',
    '--cap','24000','--record-replay')


In [ ]:
#@title Report measured performance and held-out image quality
import pandas as pd,matplotlib.pyplot as plt
rows=[]
for name,_,_ in variants:
    row=json.loads(Path(f'results/{name}/summary.json').read_text()); row['variant']=name; rows.append(row)
table=pd.DataFrame(rows)
columns=['variant','train_frames','gaussians','mapping_p50_ms','mapping_p95_ms','refine_peak_mib',
         'heldout_mean_full_psnr','heldout_mean_valid_ssim','heldout_mean_coverage','object_tracks']
compact=table[['variant','mapping_p50_ms','mapping_p95_ms','refine_peak_mib','heldout_mean_full_psnr','heldout_mean_valid_ssim']].copy()
compact['variant']=['Fusion 24k','CUDA 6k','CUDA 24k','No freeze']
compact.columns=['Variant','p50 ms','p95 ms','Peak MiB','PSNR dB','SSIM']
display(compact.round(2))
table.to_csv('results/comparison.csv',index=False)
Path('results/comparison.json').write_text(json.dumps(rows,indent=2))
fig,axes=plt.subplots(1,3,figsize=(15,3.5))
for name,_,_ in variants:
    metrics=json.loads(Path(f'results/{name}/metrics.json').read_text())
    x=[s['frame'] for s in metrics]
    axes[0].plot(x,[s['gaussians'] for s in metrics],label=name)
    axes[1].plot(x,[s['mapping_total_ms'] for s in metrics],label=name)
    old=json.loads(Path(f'results/{name}/old-view-retention.json').read_text())
    old=[s for s in old if s['heldout_frame']==3]
    axes[2].plot([s['after_train_frame'] for s in old],[s['full_psnr'] for s in old],label=name)
axes[0].set(xlabel='Training frame',ylabel='Gaussian count'); axes[0].legend(fontsize=7)
axes[1].set(xlabel='Training frame',ylabel='Fusion + refinement (ms)')
axes[2].set(xlabel='Training frame',ylabel='Fixed old heldout view PSNR (dB)')
fig.tight_layout();fig.savefig('results/real-experiment-curves.png',dpi=160);plt.show()


In [ ]:
#@title Inspect real inputs, predicted masks and held-out Gaussian rendering
from PIL import Image
from sglab.data import ManifestSequence
from sglab.server import instance_color
seq=ManifestSequence(panoptic); heldout=[i for i in range(len(seq)) if i%6==3]
indices=[heldout[0],heldout[len(heldout)//2],heldout[-1]]
fig,axes=plt.subplots(3,4,figsize=(14,9))
palette=np.array([l['color'] for l in seq.labels],dtype=np.uint8)
for row,index in enumerate(indices):
    f=seq.get(index)
    axes[row,0].imshow(f.rgb);axes[row,1].imshow(palette[f.semantic]);axes[row,2].imshow(instance_color(f.instance))
    image_path=Path('results/cuda-cap-24000')/f'heldout-{index:03}-rgb.png'
    if image_path.exists(): axes[row,3].imshow(Image.open(image_path))
    for ax in axes[row]: ax.axis('off')
for ax,title in zip(axes[0],['Real RGB-D frame','Predicted semantics','Predicted frame-local instances','Held-out Gaussian RGB']): ax.set_title(title)
fig.tight_layout();fig.savefig('results/real-data-preview.png',dpi=140);plt.show()


In [ ]:
#@title Download actual results, maps, raw predictions and replayable inputs
from google.colab import files
artifact=Path('/content/semantic-gaussian-real-results.zip')
with zipfile.ZipFile(artifact,'w',zipfile.ZIP_DEFLATED) as z:
    for folder in [Path('results'),Path('data'),Path('sglab'),Path('tests')]:
        for file in folder.rglob('*'):
            if file.is_file() and '__pycache__' not in file.parts: z.write(file,file.as_posix())
    z.write('pyproject.toml')
print('RESULT_ARTIFACT_READY',artifact,artifact.stat().st_size)
files.download(str(artifact))
